# M7-B1 — Mesures d audit (à compléter)

## 1. Disparate impact du modèle — puis investigation

DI sur prédictions et étiquettes, puis FNR/FPR et probabilité moyenne **par groupe** contre une référence construite depuis `dms_jours`.

**Variables sensibles identifiées**

**Variables sensibles directes**
- Sexe (sexe)
- Âge (age)
Ces variables sont susceptibles de créer des différences de traitement entre groupes de patients et justifient une analyse d'équité spécifique.

**Variables indirectes (proxys potentiels)**
- Département (departement): territoires différents
- Service (service): gériatrie → patients plus âgés
- Type d'admission (type_admission): accès aux soins
- Nombre de comorbidités (nb_comorbidites): corrélé à l'age

Ces variables peuvent être corrélées à certaines populations ou caractéristiques sensibles et doivent être prises en compte lors de l'interprétation des résultats.

**Variables non retenues pour l'analyse d'équité**
- patient_id (identifiant technique)
- imc
- dms_jours
- sejour_prolonge (variable cible)

In [2]:
# Analyse d'équité selon la variable sensible "sexe"

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix


# ==========================================================
# Chargement des données
# ==========================================================

project_dir = Path.cwd()

if not (project_dir / "data" / "dms_dataset.csv").exists():
    project_dir = project_dir.parent

data = pd.read_csv(
    project_dir / "data" / "dms_dataset.csv"
)

model = joblib.load(
    project_dir / "legacy" / "dms_predictor_v1.joblib"
)

# ==========================================================
# Reproduction du preprocessing historique
# ==========================================================

features = data[
    ["age", "nb_comorbidites", "imc"]
].copy()

features["sexe_bin"] = (
    data["sexe"] == "M"
).astype(int)

# ==========================================================
# Prédictions du modèle
# ==========================================================

probabilities = model.predict_proba(
    features
)[:, 1]

predictions = (
    probabilities >= 0.5
).astype(int)

labels = (
    data["sejour_prolonge"]
    .astype(int)
    .to_numpy()
)

# ==========================================================
# Référence basée sur la durée réelle
# ==========================================================

dms_reference = (
    data["dms_jours"] >= 7
).astype(int).to_numpy()

# ==========================================================
# Fonction utilitaire
# ==========================================================

def selection_rate(values):
    return np.mean(values)

# ==========================================================
# Calculs par groupe
# ==========================================================

results = []

for group, group_data in data.groupby(
    "sexe",
    sort=True
):

    indexes = group_data.index.to_numpy()

    group_predictions = predictions[indexes]
    group_labels = labels[indexes]
    group_reference = dms_reference[indexes]
    group_probabilities = probabilities[indexes]

    tn, fp, fn, tp = confusion_matrix(
        group_reference,
        group_predictions,
        labels=[0, 1]
    ).ravel()

    results.append({
        "groupe": group,
        "effectif": len(indexes),

        "taux_prediction":
            selection_rate(group_predictions),

        "taux_etiquette":
            selection_rate(group_labels),

        "taux_reference_dms_7j":
            selection_rate(group_reference),

        "probabilite_moyenne":
            np.mean(group_probabilities),

        "FNR":
            fn / (fn + tp)
            if fn + tp else np.nan,

        "FPR":
            fp / (fp + tn)
            if fp + tn else np.nan,

        "Recall":
            tp / (tp + fn)
            if tp + fn else np.nan,
    })

results = pd.DataFrame(results)

# ==========================================================
# Disparate Impact explicite F/M
# ==========================================================

female_row = (
    results
    .query("groupe == 'F'")
    .iloc[0]
)

male_row = (
    results
    .query("groupe == 'M'")
    .iloc[0]
)

di_predictions = (
    female_row["taux_prediction"]
    /
    male_row["taux_prediction"]
)

di_labels = (
    female_row["taux_etiquette"]
    /
    male_row["taux_etiquette"]
)

di_reference = (
    female_row["taux_reference_dms_7j"]
    /
    male_row["taux_reference_dms_7j"]
)

print(f"DI prédictions (F/M) : {di_predictions:.3f}")
print(f"DI étiquettes (F/M) : {di_labels:.3f}")
print(f"DI référence DMS (F/M) : {di_reference:.3f}")

display(results.round(3))

DI prédictions (F/M) : 0.291
DI étiquettes (F/M) : 0.653
DI référence DMS (F/M) : 1.013


,groupe,effectif,taux_prediction,taux_etiquette,taux_reference_dms_7j,probabilite_moyenne,FNR,FPR,Recall
0,F,5011,0.141,0.321,0.294,0.321,0.680,0.067,0.320
1,M,4989,0.486,0.491,0.290,0.491,0.169,0.345,0.831


**Conclusion du biais observé**

L'analyse du disparate impact selon le sexe met en évidence un déséquilibre important.

La référence construite à partir de la durée réelle des séjours présente un DI de 1,013, indiquant une fréquence comparable de séjours prolongés chez les femmes et les hommes.

Les étiquettes historiques présentent un DI de 0,653, révélant un écart déjà présent dans les données utilisées pour entraîner le modèle.

Les prédictions amplifient fortement cet écart avec un DI de 0,291.

L'investigation montre un taux de faux négatifs de 68 % chez les femmes contre 16,9 % chez les hommes et un recall de 32 % contre 83,1 %.

Sous l'hypothèse que le modèle sert à anticiper les séjours prolongés, les femmes apparaissent comme le groupe le plus lésé, leurs séjours prolongés étant beaucoup moins souvent détectés par le système.

In [7]:
# Calcul des DI par nb_comorbidites

for metric in [
    "taux_prediction",
    "taux_etiquette",
    "taux_reference_dms_7j"
]:

    reference_rate = (
        results_comorbidites[metric]
        .max()
    )

    results_comorbidites[
        f"DI_{metric.replace('taux_', '')}"
    ] = (
        results_comorbidites[metric]
        / reference_rate
    )

display(
    results_comorbidites.round(3)
)

,nb_comorbidites,n,taux_prediction,taux_etiquette,taux_reference_dms_7j,probabilite_moyenne,fnr_vs_dms_7j,fpr_vs_dms_7j,DI_predictions,DI_etiquettes,DI_reference_dms_7j,DI_prediction,DI_etiquette
0,0,2195,0.051,0.224,0.112,0.227,0.833,0.037,0.064,0.280,0.112,0.064,0.280
1,1,3350,0.234,0.344,0.213,0.343,0.548,0.175,0.290,0.430,0.213,0.290,0.430
2,2,2542,0.402,0.474,0.345,0.476,0.446,0.322,0.497,0.593,0.345,0.497,0.593
3,3,1259,0.554,0.589,0.495,0.585,0.310,0.421,0.687,0.737,0.495,0.687,0.737
4,4,455,0.787,0.716,0.659,0.704,0.130,0.626,0.974,0.896,0.659,0.974,0.896
5,5,161,0.807,0.708,0.801,0.704,0.155,0.656,1.000,0.885,0.801,1.000,0.885
6,6,33,0.727,0.727,0.909,0.703,0.233,0.333,0.901,0.909,0.909,0.901,0.909
7,7,5,0.800,0.800,1.000,0.739,0.200,NaN,0.991,1.000,1.000,0.991,1.000


**Analyse complémentaire : nombre de comorbidités**

Le nombre de comorbidités n'est pas une variable sensible au sens réglementaire mais constitue un facteur clinique important.

L'analyse montre une augmentation progressive du taux de séjours prolongés, de la probabilité moyenne prédite et du taux de prédictions positives lorsque le nombre de comorbidités augmente.

Cette évolution est cohérente avec l'hypothèse médicale selon laquelle les patients présentant davantage de comorbidités sont plus susceptibles de connaître des séjours prolongés.

Le principal point d'attention concerne les patients sans comorbidité, pour lesquels le taux de faux négatifs atteint 83,3 %. Le modèle identifie difficilement certains séjours prolongés lorsqu'ils ne sont pas associés à des comorbidités.

In [9]:
# ==========================================================
# Analyse par classes d'IMC
# ==========================================================

data["classe_imc"] = pd.cut(
    data["imc"],
    bins=[-np.inf, 18.5, 25, 30, np.inf],
    labels=["<18.5", "18.5-25", "25-30", ">=30"],
    right=False,
)

rows_imc = []

for group, group_data in data.groupby(
    "classe_imc",
    observed=True,
    sort=True
):

    idx = group_data.index.to_numpy()

    group_predictions = predictions[idx]
    group_labels = labels[idx]
    group_reference = dms_reference[idx]
    group_probabilities = probabilities[idx]

    tn, fp, fn, tp = confusion_matrix(
        group_reference,
        group_predictions,
        labels=[0, 1]
    ).ravel()

    rows_imc.append({
        "classe_imc": str(group),
        "effectif": len(idx),

        "taux_prediction":
            np.mean(group_predictions),

        "taux_etiquette":
            np.mean(group_labels),

        "taux_reference_dms_7j":
            np.mean(group_reference),

        "probabilite_moyenne":
            np.mean(group_probabilities),

        "recall":
            tp / (tp + fn)
            if tp + fn else np.nan,

        "fnr":
            fn / (fn + tp)
            if fn + tp else np.nan,

        "fpr":
            fp / (fp + tn)
            if fp + tn else np.nan,
    })

results_imc = pd.DataFrame(rows_imc)

# ==========================================================
# Calcul des DI
# ==========================================================

for metric in [
    "taux_prediction",
    "taux_etiquette",
    "taux_reference_dms_7j"
]:

    ref_rate = results_imc[
        metric
    ].max()

    results_imc[
        f"DI_{metric.replace('taux_', '')}"
    ] = (
        results_imc[metric]
        / ref_rate
    )

# ==========================================================
# Affichage
# ==========================================================

print("Référence : dms_jours >= 7 jours")

display(
    results_imc
    .sort_values("classe_imc")
    .round(3)
)

Référence : dms_jours >= 7 jours


,classe_imc,effectif,taux_prediction,taux_etiquette,taux_reference_dms_7j,probabilite_moyenne,recall,fnr,fpr,DI_prediction,DI_etiquette,DI_reference_dms_7j
1,18.5-25,3441,0.289,0.399,0.285,0.401,0.550,0.450,0.185,0.789,0.914,0.909
2,25-30,3721,0.305,0.405,0.289,0.402,0.548,0.452,0.207,0.834,0.928,0.922
0,<18.5,664,0.366,0.437,0.313,0.430,0.663,0.337,0.230,1.000,1.000,1.000
3,>=30,2174,0.349,0.408,0.303,0.412,0.622,0.378,0.231,0.954,0.935,0.966


**Analyse complémentaire : IMC**

L'IMC constitue une variable de santé mais n'est pas une variable sensible
au sens réglementaire.

L'analyse par classes d'IMC montre des taux de séjours prolongés relativement
proches entre groupes. Les Disparate Impacts observés restent élevés
(DI compris entre 0,789 et 1,000), traduisant une répartition relativement
homogène des prédictions.

Les taux de faux négatifs et les recalls demeurent comparables entre les
classes d'IMC, sans écart majeur suggérant une sous-détection systématique
d'un groupe particulier.

Les résultats observés semblent principalement refléter des différences
cliniques normales entre populations plutôt qu'un comportement potentiellement
inéquitable du modèle.

In [11]:
# ==========================================================
# Analyse d'équité selon l'âge
# ==========================================================

# Construction de groupes d'âge métier
data["classe_age"] = pd.cut(
    data["age"],
    bins=[-np.inf, 40, 60, 75, np.inf],
    labels=["<40", "40-59", "60-74", ">=75"],
    right=False,
)

rows_age = []

for group, group_data in data.groupby(
    "classe_age",
    observed=True,
    sort=True
):

    idx = group_data.index.to_numpy()

    group_predictions = predictions[idx]
    group_labels = labels[idx]
    group_reference = dms_reference[idx]
    group_probabilities = probabilities[idx]

    tn, fp, fn, tp = confusion_matrix(
        group_reference,
        group_predictions,
        labels=[0, 1],
    ).ravel()

    rows_age.append({
        "classe_age": str(group),

        "effectif": len(idx),

        "taux_prediction":
            np.mean(group_predictions),

        "taux_etiquette":
            np.mean(group_labels),

        "taux_reference_dms_7j":
            np.mean(group_reference),

        "probabilite_moyenne":
            np.mean(group_probabilities),

        "Recall":
            tp / (tp + fn)
            if (tp + fn) else np.nan,

        "FNR":
            fn / (fn + tp)
            if (fn + tp) else np.nan,

        "FPR":
            fp / (fp + tn)
            if (fp + tn) else np.nan,
    })

results_age = pd.DataFrame(rows_age)

# ==========================================================
# Calcul des DI
# ==========================================================

for metric in [
    "taux_prediction",
    "taux_etiquette",
    "taux_reference_dms_7j"
]:

    reference_rate = (
        results_age[metric]
        .max()
    )

    results_age[
        f"DI_{metric.replace('taux_', '')}"
    ] = (
        results_age[metric]
        / reference_rate
    )

# ==========================================================
# Affichage final
# ==========================================================

print("Référence : dms_jours >= 7 jours")

display(
    results_age
    .sort_values("classe_age")
    .round(3)
)

Référence : dms_jours >= 7 jours


,classe_age,effectif,taux_prediction,taux_etiquette,taux_reference_dms_7j,probabilite_moyenne,Recall,FNR,FPR,DI_prediction,DI_etiquette,DI_reference_dms_7j
1,40-59,2668,0.249,0.380,0.256,0.379,0.474,0.526,0.172,0.430,0.675,0.557
2,60-74,1881,0.390,0.461,0.332,0.462,0.590,0.410,0.290,0.672,0.818,0.722
0,<40,2893,0.087,0.254,0.152,0.255,0.335,0.665,0.042,0.150,0.451,0.330
3,>=75,2558,0.580,0.564,0.459,0.562,0.711,0.289,0.468,1.000,1.000,1.000


**Analyse selon l'âge**

L'analyse du disparate impact selon l'âge montre que les taux de prédiction
positive augmentent fortement avec l'âge.

Les patients de moins de 40 ans présentent un DI de 0,150 par rapport aux
patients de 75 ans et plus, qui constituent le groupe de référence.

Cet écart s'explique toutefois en partie par la réalité observée dans les
données : les patients âgés connaissent effectivement davantage de séjours
prolongés (45,9 % contre 15,2 % chez les moins de 40 ans).

L'investigation montre néanmoins une dégradation des performances de détection
chez les patients les plus jeunes. Leur taux de faux négatifs atteint 66,5 %
contre 28,9 % chez les patients de 75 ans et plus.

Le modèle semble donc moins performant pour détecter les séjours prolongés
chez les patients jeunes, même si les différences observées reflètent en partie
des écarts réels de durée de séjour.

## 2. Ressources (psutil)

In [33]:
import os
import time
import psutil
import joblib
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [13]:
# ==========================================================
# Audit ressources du modèle historique
# ==========================================================

import os
import time

import pandas as pd
import psutil

# Processus Python courant
process = psutil.Process(os.getpid())

# Localisation du modèle
model_path = (
    project_dir
    / "legacy"
    / "dms_predictor_v1.joblib"
)

# ==========================================================
# Mesures de référence
# ==========================================================

rss_initial_mb = (
    process.memory_info().rss
    / (1024 ** 2)
)

model_size_mb = (
    model_path.stat().st_size
    / (1024 ** 2)
)

# ==========================================================
# Mesures d'inférence
# ==========================================================

resource_results = []

for sample_size in [100, 1_000, 10_000]:

    # Génération du lot de test
    sample_features = (
        pd.concat(
            [features] * (
                (sample_size + len(features) - 1)
                // len(features)
            ),
            ignore_index=True
        )
        .iloc[:sample_size]
    )

    rss_before_mb = (
        process.memory_info().rss
        / (1024 ** 2)
    )

    start_time = time.perf_counter()

    _ = model.predict_proba(
        sample_features
    )

    elapsed_ms = (
        time.perf_counter()
        - start_time
    ) * 1000

    rss_after_mb = (
        process.memory_info().rss
        / (1024 ** 2)
    )

    resource_results.append({
        "volume": sample_size,
        "temps_inference_ms":
            elapsed_ms,

        "temps_par_ligne_us":
            (elapsed_ms * 1000)
            / sample_size,

        "rss_avant_mb":
            rss_before_mb,

        "rss_apres_mb":
            rss_after_mb,

        "variation_rss_mb":
            rss_after_mb - rss_before_mb,

        "taille_modele_mb":
            model_size_mb,
    })

# ==========================================================
# Résultats
# ==========================================================

resource_results = pd.DataFrame(
    resource_results
)

print(
    f"RSS initiale : "
    f"{rss_initial_mb:.2f} MB"
)

print(
    f"Taille du modèle : "
    f"{model_size_mb:.2f} MB"
)

display(
    resource_results.round(4)
)

RSS initiale : 50.89 MB
Taille du modèle : 4.73 MB


,volume,temps_inference_ms,temps_par_ligne_us,rss_avant_mb,rss_apres_mb,variation_rss_mb,taille_modele_mb
0,100,68.3177,683.1770,52.7578,60.1406,7.3828,4.7268
1,1000,16.6562,16.6562,60.1523,60.2383,0.0859,4.7268
2,10000,59.6162,5.9616,60.2383,60.5977,0.3594,4.7268


**Interprétation**

 
Le modèle occupe environ **4,73 MB** sur disque et présente une empreinte mémoire relativement faible à l'échelle de l'environnement d'exécution.
 
Les temps d'inférence restent inférieurs à 100 ms même pour un volume de 10 000 observations, ce qui indique un niveau de performance satisfaisant pour les volumes testés.
 
La mémoire consommée reste stable lors de l'augmentation du nombre d'observations, avec une variation inférieure à 1 MB entre 1 000 et 10 000 lignes.
 
Le temps moyen par observation diminue lorsque la taille des lots augmente :
 
- 683 µs par ligne pour 100 observations ;
- 16,7 µs par ligne pour 1 000 observations ;
- 6 µs par ligne pour 10 000 observations.
 
Cette évolution suggère que le coût fixe de l'appel au modèle est amorti lorsque le traitement est réalisé par lot.

Aucun risque critique de performance ou de consommation mémoire n'a été identifié sur les volumes testés.
 
Le principal point d'attention du volet ressources concerne davantage la comparaison avec des alternatives plus sobres (par exemple une régression logistique) que les performances intrinsèques du modèle sur le périmètre observé.

## 3. Comparaison à 2 alternatives

In [16]:
# ==========================================================
# Comparaison du modèle historique avec deux alternatives
# ==========================================================

import pickle
import time

import os

os.environ["LOKY_MAX_CPU_COUNT"] = "8"

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ==========================================================
# Préparation des données
# ==========================================================

X = features.copy()
y = data["sejour_prolonge"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# ==========================================================
# Modèles comparés
# ==========================================================

models = {
    "Random Forest (legacy)": RandomForestClassifier(
        n_estimators=60,
        max_depth=10,
        random_state=0,
    ),

    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=1000,
            random_state=42,
        ),
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        random_state=42,
    ),
}

# ==========================================================
# Mesures
# ==========================================================

results = []

for model_name, candidate_model in models.items():

    # -------------------------
    # Entraînement
    # -------------------------

    train_start = time.perf_counter()

    candidate_model.fit(
        X_train,
        y_train,
    )

    train_time_ms = (
        time.perf_counter() - train_start
    ) * 1000

    # -------------------------
    # Inférence
    # -------------------------

    inference_start = time.perf_counter()

    probabilities = candidate_model.predict_proba(
        X_test
    )[:, 1]

    inference_time_ms = (
        time.perf_counter() - inference_start
    ) * 1000

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    # -------------------------
    # Taille du modèle
    # -------------------------

    model_size_mb = (
        len(
            pickle.dumps(
                candidate_model
            )
        )
        / (1024 ** 2)
    )

    # -------------------------
    # Métriques
    # -------------------------

    results.append({
        "Modèle": model_name,

        "Accuracy":
            accuracy_score(
                y_test,
                predictions,
            ),

        "Balanced Accuracy":
            balanced_accuracy_score(
                y_test,
                predictions,
            ),

        "ROC-AUC":
            roc_auc_score(
                y_test,
                probabilities,
            ),

        "Temps entraînement (ms)":
            train_time_ms,

        "Temps inférence (ms)":
            inference_time_ms,

        "Taille modèle (MB)":
            model_size_mb,
    })

comparison_results = (
    pd.DataFrame(results)
    .sort_values(
        "ROC-AUC",
        ascending=False,
    )
)

display(
    comparison_results.round(4)
)

,Modèle,Accuracy,Balanced Accuracy,ROC-AUC,Temps entraînement (ms),Temps inférence (ms),Taille modèle (MB)
1,Logistic Regression,0.691,0.6596,0.7364,10.9711,1.7073,0.0013
2,HistGradientBoosting,0.678,0.6473,0.7199,755.7818,10.2935,0.3431
0,Random Forest (legacy),0.676,0.6429,0.7195,489.1668,19.4647,4.3748


**Synthèse:**

La comparaison entre le modèle historique et deux alternatives montre
que la régression logistique obtient les meilleures performances tout en
présentant le coût technique le plus faible.

Le modèle Random Forest historique est plus volumineux, plus lent à
entraîner et plus lent en inférence, sans gain de performance observable
sur le jeu de test évalué.

Du point de vue de la sobriété numérique, la régression logistique
constitue donc l'alternative la plus efficiente parmi les modèles testés.

L'objectif de cet audit n'est toutefois pas de recommander un
remplacement du modèle existant mais de documenter le rapport entre
performance et coût des différentes solutions observées.